# RQ3 modeling pipeline (age strand + disability strand)

Two modeling tasks per strand:
1. **Overall activity level** -- year x borough x group, predicting the
   inactive/fairly_active/active three-way share 
2. **Activity-specific participation** -- year x borough x group x activity,
   predicting participation for each individual activity, used to answer both
   "will this group participate in this activity" and "which activity does this
   group prefer most".



## 0. Setup

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
ALR_EPSILON = 1e-6
AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed") 

## 1. Shared functions


In [2]:
def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


In [ ]:
def score_composition(actual, predicted, target_names):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual).all(axis=1) & np.isfinite(predicted).all(axis=1)
    actual, predicted = actual[valid], predicted[valid]
    row = {
        'observations': len(actual),
        'total_variation': np.mean(.5 * np.abs(actual - predicted).sum(axis=1)),
        'overall_mae': np.mean(np.abs(actual - predicted)),
        'overall_rmse': np.sqrt(np.mean((actual - predicted) ** 2)),
    }
    for index, target in enumerate(target_names):
        row[f'{target}_mae'] = mean_absolute_error(actual[:, index], predicted[:, index])
        row[f'{target}_rmse'] = np.sqrt(mean_squared_error(actual[:, index], predicted[:, index]))
        row[f'{target}_r2'] = r2_score(actual[:, index], predicted[:, index])
    return row


def score_single_rate(actual, predicted, target_name):
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    return {
        'observations': len(actual),
        f'{target_name}_mae': mean_absolute_error(actual, predicted),
        f'{target_name}_rmse': np.sqrt(mean_squared_error(actual, predicted)),
        f'{target_name}_r2': r2_score(actual, predicted),
    }


In [4]:
def add_lag_features(frame, panel_keys, value_cols, lag=1):
    prepared = frame.sort_values(panel_keys + ['year']).reset_index(drop=True)
    grouped = prepared.groupby(panel_keys, sort=False)
    for column in value_cols:
        prepared[f'{column}_lag{lag}'] = grouped[column].shift(lag)
    prepared['time_trend'] = prepared['year'] / prepared['year'].max()
    return prepared


def naive_predict(frame, lag_cols):
    return frame[lag_cols].to_numpy()


In [5]:
def parameter_candidates(model_name):
    if model_name == 'Ridge Regression':
        return [{'alpha': .1}, {'alpha': 1.0}, {'alpha': 10.0}, {'alpha': 100.0}]
    if model_name == 'Random Forest':
        return [
            {'n_estimators': 160, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': .5},
            {'n_estimators': 160, 'max_depth': 10, 'min_samples_leaf': 8, 'max_features': .5},
            {'n_estimators': 220, 'max_depth': 8, 'min_samples_leaf': 12, 'max_features': .8},
        ]
    return [
        {'n_estimators': 120, 'learning_rate': .05, 'max_depth': 2, 'min_samples_leaf': 15},
        {'n_estimators': 160, 'learning_rate': .03, 'max_depth': 2, 'min_samples_leaf': 20},
    ]


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_name == 'Ridge Regression':
        numeric_steps.append(('scale', StandardScaler()))
    preprocess = ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ])
    if model_name == 'Ridge Regression':
        estimator = Ridge(**parameters)
    elif model_name == 'Random Forest':
        estimator = RandomForestRegressor(**parameters, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        base = GradientBoostingRegressor(**parameters, random_state=RANDOM_STATE, loss='huber')
        estimator = MultiOutputRegressor(base) if multi_output else base
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


## 2. Task type 1: three-way composition (inactive / fairly_active / active)


In [ ]:
def run_composition_task(frame, panel_keys, target_cols, weight_col,
                          val_year=7, test_year=8):
    """panel_keys: the grouping columns that define one time series, e.g.
    ['LA_2023', 'age_group'] (overall level) or
    ['LA_2023', 'disability_group', 'activity'] (activity-specific level)."""
    prepared = add_lag_features(frame, panel_keys, target_cols, lag=1)
    lag_cols = [f'{c}_lag1' for c in target_cols]
    categorical = list(panel_keys)
    numeric = lag_cols + ['time_trend']

    train = prepared[prepared['year'] < val_year].dropna(subset=target_cols)
    val = prepared[prepared['year'] == val_year].dropna(subset=target_cols + lag_cols)
    trainval = prepared[prepared['year'] < test_year].dropna(subset=target_cols)
    test = prepared[prepared['year'] == test_year].dropna(subset=target_cols + lag_cols)

    results = {}
    for label, split in [('validation', val), ('test', test)]:
        pred = naive_predict(split, lag_cols)
        results[('Naive baseline', label)] = score_composition(split[target_cols], pred, target_cols)

    for model_name in ['Ridge Regression', 'Random Forest', 'Gradient Boosting']:
        best_params, best_score = None, np.inf
        for params in parameter_candidates(model_name):
            model = build_model(model_name, params, numeric, categorical, multi_output=True)
            w = train[weight_col].to_numpy() if weight_col else None
            fit_kwargs = {'model__sample_weight': w} if (w is not None and model_name != 'Ridge Regression') else {}
            model.fit(train[numeric + categorical], shares_to_alr(train[target_cols]), **fit_kwargs)
            pred = alr_to_shares(model.predict(val[numeric + categorical]))
            score = score_composition(val[target_cols], pred, target_cols)
            if score['overall_mae'] < best_score:
                best_score, best_params = score['overall_mae'], params

        final_model = build_model(model_name, best_params, numeric, categorical, multi_output=True)
        w = trainval[weight_col].to_numpy() if weight_col else None
        fit_kwargs = {'model__sample_weight': w} if (w is not None and model_name != 'Ridge Regression') else {}
        final_model.fit(trainval[numeric + categorical], shares_to_alr(trainval[target_cols]), **fit_kwargs)
        pred_test = alr_to_shares(final_model.predict(test[numeric + categorical]))
        results[(model_name, 'test')] = score_composition(test[target_cols], pred_test, target_cols)
        results[(model_name, 'best_params')] = best_params
        results[(model_name, 'fitted_model')] = final_model

    return results, prepared


### 2.1 Age strand -- overall activity level

In [15]:
age_overall = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_overall['LA_2023'] = age_overall['LA_2023'].astype('Int64').astype(str)

age_overall_targets = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']
age_overall_results, age_overall_panel = run_composition_task(
    age_overall,
    panel_keys=['LA_2023', 'age_group'],
    target_cols=age_overall_targets,
    weight_col='weighted_n_overall_activity_level',
)

for key, value in age_overall_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

('Naive baseline', 'test') {'observations': 256, 'total_variation': np.float64(0.1377), 'overall_mae': np.float64(0.0918), 'overall_rmse': np.float64(0.1432), 'overall_inactive_rate_mae': 0.1034, 'overall_inactive_rate_rmse': np.float64(0.162), 'overall_inactive_rate_r2': 0.0572, 'overall_fairly_active_rate_mae': 0.0616, 'overall_fairly_active_rate_rmse': np.float64(0.085), 'overall_fairly_active_rate_r2': -1.0996, 'overall_active_rate_mae': 0.1104, 'overall_active_rate_rmse': np.float64(0.1674), 'overall_active_rate_r2': 0.0992}
('Ridge Regression', 'test') {'observations': 256, 'total_variation': np.float64(0.1317), 'overall_mae': np.float64(0.0878), 'overall_rmse': np.float64(0.144), 'overall_inactive_rate_mae': 0.108, 'overall_inactive_rate_rmse': np.float64(0.1767), 'overall_inactive_rate_r2': -0.122, 'overall_fairly_active_rate_mae': 0.0555, 'overall_fairly_active_rate_rmse': np.float64(0.0791), 'overall_fairly_active_rate_r2': -0.8169, 'overall_active_rate_mae': 0.0998, 'overall

### 2.2 Disability strand -- overall activity level



In [16]:
dis_overall = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
dis_overall['LA_2023'] = dis_overall['LA_2023'].astype('Int64').astype(str)

dis_overall_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_overall_results, dis_overall_panel = run_composition_task(
    dis_overall,
    panel_keys=['LA_2023', 'disability_group'],
    target_cols=dis_overall_targets,
    weight_col='weighted_n',
)

for key, value in dis_overall_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

('Naive baseline', 'test') {'observations': 510, 'total_variation': np.float64(0.2606), 'overall_mae': np.float64(0.1738), 'overall_rmse': np.float64(0.2539), 'inactive_rate_mae': 0.2027, 'inactive_rate_rmse': np.float64(0.2765), 'inactive_rate_r2': -0.7163, 'fairly_active_rate_mae': 0.1044, 'fairly_active_rate_rmse': np.float64(0.1711), 'fairly_active_rate_r2': -1.3121, 'active_rate_mae': 0.2141, 'active_rate_rmse': np.float64(0.296), 'active_rate_r2': -0.8359}
('Ridge Regression', 'test') {'observations': 510, 'total_variation': np.float64(0.2199), 'overall_mae': np.float64(0.1466), 'overall_rmse': np.float64(0.2043), 'inactive_rate_mae': 0.1566, 'inactive_rate_rmse': np.float64(0.2081), 'inactive_rate_r2': 0.0274, 'fairly_active_rate_mae': 0.0901, 'fairly_active_rate_rmse': np.float64(0.1438), 'fairly_active_rate_r2': -0.6326, 'active_rate_mae': 0.1932, 'active_rate_rmse': np.float64(0.2473), 'active_rate_r2': -0.2815}
('Random Forest', 'test') {'observations': 510, 'total_variation

### 2.3 Disability strand -- activity-specific MEMS7GR tiers



In [17]:
dis_level = pd.read_csv(DATA_DIR / 'RQ3_borough_disability_all_years.csv')
dis_level['LA_2023'] = dis_level['LA_2023'].astype('Int64').astype(str)

dis_level_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_level_results, dis_level_panel = run_composition_task(
    dis_level,
    panel_keys=['LA_2023', 'disability_group', 'activity'],
    target_cols=dis_level_targets,
    weight_col='weighted_n',
)

for key, value in dis_level_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})


('Naive baseline', 'test') {'observations': 62886, 'total_variation': np.float64(0.0118), 'overall_mae': np.float64(0.0079), 'overall_rmse': np.float64(0.0458), 'inactive_rate_mae': 0.0107, 'inactive_rate_rmse': np.float64(0.055), 'inactive_rate_r2': 0.0778, 'fairly_active_rate_mae': 0.0057, 'fairly_active_rate_rmse': np.float64(0.0362), 'fairly_active_rate_r2': -0.6515, 'active_rate_mae': 0.0071, 'active_rate_rmse': np.float64(0.0443), 'active_rate_r2': -0.1137}
('Ridge Regression', 'test') {'observations': 62886, 'total_variation': np.float64(0.0097), 'overall_mae': np.float64(0.0064), 'overall_rmse': np.float64(0.04), 'inactive_rate_mae': 0.0096, 'inactive_rate_rmse': np.float64(0.0514), 'inactive_rate_r2': 0.1951, 'fairly_active_rate_mae': 0.0042, 'fairly_active_rate_rmse': np.float64(0.0279), 'fairly_active_rate_r2': 0.0191, 'active_rate_mae': 0.0055, 'active_rate_rmse': np.float64(0.0372), 'active_rate_r2': 0.2159}
('Random Forest', 'test') {'observations': 62886, 'total_variatio

## 3. Task type 2: single participation rate (MONTHS_12 / DAYS10P60GR)



In [18]:
def run_single_rate_task(frame, panel_keys, target_col, weight_col,
                          val_year=7, test_year=8):
    prepared = add_lag_features(frame, panel_keys, [target_col], lag=1)
    lag_col = f'{target_col}_lag1'
    categorical = list(panel_keys)
    numeric = [lag_col, 'time_trend']

    train = prepared[prepared['year'] < val_year].dropna(subset=[target_col])
    val = prepared[prepared['year'] == val_year].dropna(subset=[target_col, lag_col])
    trainval = prepared[prepared['year'] < test_year].dropna(subset=[target_col])
    test = prepared[prepared['year'] == test_year].dropna(subset=[target_col, lag_col])

    results = {}
    for label, split in [('validation', val), ('test', test)]:
        pred = naive_predict(split, [lag_col])
        results[('Naive baseline', label)] = score_single_rate(split[target_col], pred, target_col)

    for model_name in ['Ridge Regression', 'Random Forest', 'Gradient Boosting']:
        best_params, best_score = None, np.inf
        for params in parameter_candidates(model_name):
            model = build_model(model_name, params, numeric, categorical, multi_output=False)
            w = train[weight_col].to_numpy() if weight_col else None
            fit_kwargs = {'model__sample_weight': w} if (w is not None and model_name != 'Ridge Regression') else {}
            model.fit(train[numeric + categorical], train[target_col], **fit_kwargs)
            pred = np.clip(model.predict(val[numeric + categorical]), 0, 1)
            score = score_single_rate(val[target_col], pred, target_col)
            if score[f'{target_col}_mae'] < best_score:
                best_score, best_params = score[f'{target_col}_mae'], params

        final_model = build_model(model_name, best_params, numeric, categorical, multi_output=False)
        w = trainval[weight_col].to_numpy() if weight_col else None
        fit_kwargs = {'model__sample_weight': w} if (w is not None and model_name != 'Ridge Regression') else {}
        final_model.fit(trainval[numeric + categorical], trainval[target_col], **fit_kwargs)
        pred_test = np.clip(final_model.predict(test[numeric + categorical]), 0, 1)
        results[(model_name, 'test')] = score_single_rate(test[target_col], pred_test, target_col)
        results[(model_name, 'best_params')] = best_params
        results[(model_name, 'fitted_model')] = final_model

    return results, prepared, test


### 3.1 Disability strand -- MONTHS_12 and DAYS10P60GR



In [19]:
dis_dm = pd.read_csv(DATA_DIR / 'RQ3_borough_disability_days_months_all_years.csv')
dis_dm['LA_2023'] = dis_dm['LA_2023'].astype('Int64').astype(str)

months12_results, months12_panel, months12_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_MONTHS_12', weight_col='weighted_n_MONTHS_12',
)
days_results, days_panel, days_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_DAYS10P60GR', weight_col='weighted_n_DAYS10P60GR',
)

for label, results in [('MONTHS_12', months12_results), ('DAYS10P60GR', days_results)]:
    print(f'--- {label} ---')
    for key, value in results.items():
        if key[1] == 'test':
            print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})


--- MONTHS_12 ---
('Naive baseline', 'test') {'observations': 62886, 'participation_MONTHS_12_mae': 0.0237, 'participation_MONTHS_12_rmse': np.float64(0.0836), 'participation_MONTHS_12_r2': 0.3122}
('Ridge Regression', 'test') {'observations': 62886, 'participation_MONTHS_12_mae': 0.0213, 'participation_MONTHS_12_rmse': np.float64(0.0608), 'participation_MONTHS_12_r2': 0.6361}
('Random Forest', 'test') {'observations': 62886, 'participation_MONTHS_12_mae': 0.0245, 'participation_MONTHS_12_rmse': np.float64(0.0615), 'participation_MONTHS_12_r2': 0.6279}
('Gradient Boosting', 'test') {'observations': 62886, 'participation_MONTHS_12_mae': 0.0241, 'participation_MONTHS_12_rmse': np.float64(0.0744), 'participation_MONTHS_12_r2': 0.455}
--- DAYS10P60GR ---
('Naive baseline', 'test') {'observations': 62886, 'participation_DAYS10P60GR_mae': 0.0113, 'participation_DAYS10P60GR_rmse': np.float64(0.0569), 'participation_DAYS10P60GR_r2': 0.1849}
('Ridge Regression', 'test') {'observations': 62886, 

### 3.2 Age strand -- months12_rate and days10p60gr_rate from the complete table



In [22]:
age_complete = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
age_complete['LA_2023'] = age_complete['LA_2023'].astype('Int64').astype(str)

age_months12_results, age_months12_panel, age_months12_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='months12_rate', weight_col='weighted_n_months12',
)
age_days_results, age_days_panel, age_days_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='days10p60gr_rate', weight_col='weighted_n_days10p60gr',
)

age_complete_targets = ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate']
age_level_results, age_level_panel = run_composition_task(
    age_complete, panel_keys=['LA_2023', 'age_group', 'activity_suffix'],
    target_cols=age_complete_targets, weight_col='weighted_n_activity_level',
)

## 4. Extracting "most preferred activity"



In [23]:
def most_preferred_activity(fitted_model, panel, numeric_cols, categorical_cols, id_cols,
                             activity_col, top_n=1):
    working = panel.dropna(subset=numeric_cols + categorical_cols).copy()
    working['predicted_participation'] = np.clip(
        fitted_model.predict(working[numeric_cols + categorical_cols]), 0, 1
    )
    ranked = working.sort_values(
        id_cols + ['predicted_participation'], ascending=[True] * len(id_cols) + [False]
    )
    return ranked.groupby(id_cols, as_index=False).head(top_n)[
        id_cols + [activity_col, 'predicted_participation']
    ]


best_disability_model = months12_results[('Random Forest', 'fitted_model')]
top_activity_by_disability_group = most_preferred_activity(
    best_disability_model, months12_test,
    numeric_cols=['participation_MONTHS_12_lag1', 'time_trend'],
    categorical_cols=['LA_2023', 'disability_group', 'activity'],
    id_cols=['LA_2023', 'disability_group'],
    activity_col='activity',
)
top_activity_by_disability_group.head(10)


,LA_2023,disability_group,activity,predicted_participation
950,107,disty1,WALKTRAV_B02,0.516233
1014,107,disty10,ACTTRAV_C03,0.474420
2013,107,disty11,ACTTRAV_C03,0.479878
3012,107,disty12,ACTTRAV_C03,0.574784
4011,107,disty13,ACTTRAV_C03,0.490765
5010,107,disty2,ACTTRAV_C03,0.539910
6009,107,disty3,ACTTRAV_C03,0.419377
7008,107,disty4,ACTTRAV_C03,0.536523
8007,107,disty5,ACTTRAV_C03,0.610774
9006,107,disty6,ACTTRAV_C03,0.520881


## 5. Summary table



In [24]:
def collect_summary(results_dict, task_name):
    rows = []
    for (model_name, split), value in results_dict.items():
        if split == 'test' and isinstance(value, dict):
            rows.append({'task': task_name, 'model': model_name, **value})
    return pd.DataFrame(rows)

summary = pd.concat([
    collect_summary(age_overall_results, 'age_overall_level'),
    collect_summary(dis_overall_results, 'disability_overall_level'),
    collect_summary(dis_level_results, 'disability_activity_level'),
    collect_summary(months12_results, 'disability_months12'),
    collect_summary(days_results, 'disability_days10p60gr'),
    collect_summary(age_months12_results, 'age_months12'),
    collect_summary(age_days_results, 'age_days10p60gr'),
    collect_summary(age_level_results, 'age_activity_level'),
], ignore_index=True)

summary


,task,model,observations,total_variation,overall_mae,overall_rmse,overall_inactive_rate_mae,overall_inactive_rate_rmse,overall_inactive_rate_r2,overall_fairly_active_rate_mae,...,days10p60gr_rate_r2,activity_inactive_rate_mae,activity_inactive_rate_rmse,activity_inactive_rate_r2,activity_fairly_active_rate_mae,activity_fairly_active_rate_rmse,activity_fairly_active_rate_r2,activity_active_rate_mae,activity_active_rate_rmse,activity_active_rate_r2
0,age_overall_level,Naive baseline,256,0.137717,0.091811,0.143164,0.103420,0.162009,0.057195,0.061627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,age_overall_level,Ridge Regression,256,0.131694,0.087796,0.144036,0.108024,0.176734,-0.121973,0.055533,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,age_overall_level,Random Forest,256,0.123361,0.082241,0.136262,0.098600,0.166022,0.009918,0.048582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,age_overall_level,Gradient Boosting,256,0.109442,0.072962,0.110913,0.086133,0.127667,0.414534,0.042517,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,disability_overall_level,Naive baseline,510,0.260630,0.173753,0.253874,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,disability_overall_level,Ridge Regression,510,0.219942,0.146628,0.204260,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,disability_overall_level,Random Forest,510,0.210276,0.140184,0.208160,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,disability_overall_level,Gradient Boosting,510,0.192813,0.128542,0.183067,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,disability_activity_level,Naive baseline,62886,0.011798,0.007866,0.045802,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,disability_activity_level,Ridge Regression,62886,0.009668,0.006445,0.039991,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import os
import pickle

output_dir = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. summary table
summary.to_csv(os.path.join(output_dir, 'summary_full.csv'), index=False)

# 2. most preferred activity table
top_activity_by_disability_group.to_csv(
    os.path.join(output_dir, 'top_activity_by_disability_group.csv'), index=False
)

# 3. every fitted model from every task
all_results = {
    'age_overall': age_overall_results,
    'dis_overall': dis_overall_results,
    'dis_level': dis_level_results,
    'months12': months12_results,
    'days': days_results,
    'age_months12': age_months12_results,
    'age_days': age_days_results,
    'age_level': age_level_results,
}

fitted_models = {}
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'fitted_model':
            fitted_models[f'{task_name}__{model_name}'] = value

with open(os.path.join(output_dir, 'fitted_models.pkl'), 'wb') as f:
    pickle.dump(fitted_models, f)

# 4. best_params for every task/model
best_params_rows = []
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'best_params':
            best_params_rows.append({'task': task_name, 'model': model_name, **value})

import pandas as pd
pd.DataFrame(best_params_rows).to_csv(
    os.path.join(output_dir, 'best_hyperparameters.csv'), index=False
)

print('Saved to', output_dir)
print('- summary_full.csv')
print('- top_activity_by_disability_group.csv')
print('- fitted_models.pkl', f'({len(fitted_models)} models)')
print('- best_hyperparameters.csv')

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs
- summary_full.csv
- top_activity_by_disability_group.csv
- fitted_models.pkl (24 models)
- best_hyperparameters.csv
